# Bronze Ingest: Wikipedia Clickstream

Downloads a monthly Wikipedia Clickstream dataset (English Wikipedia by
default) and persists it as a managed Spark table, unmodified from source.
This is the `bronze` layer.

Each row is a `(prev, curr, type, n)` tuple: `n` requests arrived at page
`curr` with referrer `prev` during the month, where `type` is `link`
(internal wiki link), `external`, or `other`. See
https://meta.wikimedia.org/wiki/Research:Wikipedia_clickstream for the
full schema description.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and `mode("overwrite")`,
so re-running just refreshes the table. Spark's CSV reader decompresses
`.gz` automatically, so no manual unzip step is needed.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

Change `WIKI` / `MONTH` to pull a different edition or month without
touching the logic below. Available editions/months are listed at
https://dumps.wikimedia.org/other/clickstream/

In [ ]:
DATABASE_NAME = "wikipedia"
WIKI = "enwiki"
MONTH = "2026-07"
TABLE_NAME = f"clickstream_{WIKI}"
SOURCE_URL = f"https://dumps.wikimedia.org/other/clickstream/{MONTH}/clickstream-{WIKI}-{MONTH}.tsv.gz"
LOCAL_FILE = f"/tmp/clickstream-{WIKI}-{MONTH}.tsv.gz"

## Download source file

In [ ]:
import urllib.request

urllib.request.urlretrieve(SOURCE_URL, LOCAL_FILE)

## Load dataset

Tab-separated, no header, 4 columns: `prev`, `curr`, `type`, `n`.

In [ ]:
df = (
    spark.read
    .option("sep", "\t")
    .option("header", "false")
    .csv(LOCAL_FILE)
    .toDF("prev", "curr", "type", "n")
    .withColumn("n", F.col("n").cast("long"))
)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Wikipedia Clickstream datasets used for analytics demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)